# Coralscapes -> YOLO Semantic Segmentation Format

Downloads the [Coralscapes](https://huggingface.co/datasets/EPFL-ECEO/coralscapes) dataset from Hugging Face and
converts it into the directory layout + dataset YAML expected by Ultralytics YOLO26 `semantic` task
(see `ultralytics/cfg/datasets/cityscapes.yaml` for the reference format).

In [1]:
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from tqdm.auto import tqdm

REPO_ID = "EPFL-ECEO/coralscapes"
OUTPUT_DIR = Path("../../coralscapes_yolo").resolve()
SPLIT_MAP = {"train": "train", "validation": "val", "test": "test"}  # HF split name -> YOLO split name

for split in SPLIT_MAP.values():
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "masks" / split).mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

WindowsPath('C:/Users/jordan/Documents/GitHub/coralscapes_yolo')

## Class mapping

`id2label.json` maps raw mask pixel values `1..39` to class names (pixel value `0` means "unlabeled" / no
annotation and is not itself a class). YOLO's `SemanticDataset` needs contiguous `0..N-1` ids, so classes are
shifted down by one and raw id `0` is remapped to the ignore label (`255`) via `label_mapping`.

In [2]:
id2label_path = hf_hub_download(REPO_ID, "id2label.json", repo_type="dataset")
label2color_path = hf_hub_download(REPO_ID, "label2color.json", repo_type="dataset")

with open(id2label_path) as f:
    id2label = json.load(f)  # {"1": "seagrass", ..., "39": "dead clam"}
with open(label2color_path) as f:
    label2color = json.load(f)  # {class_name: [r, g, b]}

sorted_ids = sorted(int(k) for k in id2label)  # [1, 2, ..., 39]
names = {i: id2label[str(src_id)] for i, src_id in enumerate(sorted_ids)}
label_mapping = {0: "ignore_label", **{src_id: i for i, src_id in enumerate(sorted_ids)}}

print(f"{len(names)} classes")
names

39 classes


{0: 'seagrass',
 1: 'trash',
 2: 'other coral dead',
 3: 'other coral bleached',
 4: 'sand',
 5: 'other coral alive',
 6: 'human',
 7: 'transect tools',
 8: 'fish',
 9: 'algae covered substrate',
 10: 'other animal',
 11: 'unknown hard substrate',
 12: 'background',
 13: 'dark',
 14: 'transect line',
 15: 'massive/meandering bleached',
 16: 'massive/meandering alive',
 17: 'rubble',
 18: 'branching bleached',
 19: 'branching dead',
 20: 'millepora',
 21: 'branching alive',
 22: 'massive/meandering dead',
 23: 'clam',
 24: 'acropora alive',
 25: 'sea cucumber',
 26: 'turbinaria',
 27: 'table acropora alive',
 28: 'sponge',
 29: 'anemone',
 30: 'pocillopora alive',
 31: 'table acropora dead',
 32: 'meandering bleached',
 33: 'stylophora alive',
 34: 'sea urchin',
 35: 'meandering alive',
 36: 'meandering dead',
 37: 'crown of thorn',
 38: 'dead clam'}

## Download parquet shards and write YOLO image/mask pairs

Each row's `image`/`label` fields are decoded from their stored bytes and re-saved as PNGs. The mask is kept at its
raw pixel values (`0..39`); `label_mapping` in the dataset YAML performs the id shift at train/val time, so no pixel
remapping is needed here.

In [ ]:
api = HfApi()
all_files = api.list_repo_files(REPO_ID, repo_type="dataset")


def shards_for(hf_split):
    prefix = f"data/{hf_split}-"
    return sorted(f for f in all_files if f.startswith(prefix))


def convert_split(hf_split, yolo_split):
    image_dir = OUTPUT_DIR / "images" / yolo_split
    mask_dir = OUTPUT_DIR / "masks" / yolo_split
    n = 0
    for shard in tqdm(shards_for(hf_split), desc=f"{hf_split} shards"):
        shard_path = hf_hub_download(REPO_ID, shard, repo_type="dataset")
        df = pd.read_parquet(shard_path)
        for _, row in tqdm(df.iterrows(), total=len(df), desc=shard, leave=False):
            image = Image.open(io.BytesIO(row["image"]["bytes"])).convert("RGB")
            mask = Image.open(io.BytesIO(row["label"]["bytes"])).convert("L")

            stem = Path(row["image"]["path"]).stem.replace("_leftImg8bit", "")
            image.save(image_dir / f"{stem}.png")
            mask.save(mask_dir / f"{stem}.png")
            n += 1
    return n


counts = {yolo_split: convert_split(hf_split, yolo_split) for hf_split, yolo_split in SPLIT_MAP.items()}
counts

## Write dataset YAML

In [4]:
with open(OUTPUT_DIR / "id2label.json", "w") as f:
    json.dump(id2label, f, indent=2)
with open(OUTPUT_DIR / "label2color.json", "w") as f:
    json.dump(label2color, f, indent=2)

data_yaml = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "masks_dir": "masks",
    "names": names,
    "label_mapping": label_mapping,
}

yaml_path = OUTPUT_DIR / "coralscapes.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False, default_flow_style=False)

print(f"Wrote {yaml_path}")

Wrote C:\Users\jordan\Documents\GitHub\coralscapes_yolo\coralscapes.yaml


## Sanity check

In [5]:
for yolo_split in SPLIT_MAP.values():
    n_images = len(list((OUTPUT_DIR / "images" / yolo_split).glob("*.png")))
    n_masks = len(list((OUTPUT_DIR / "masks" / yolo_split).glob("*.png")))
    print(f"{yolo_split}: {n_images} images, {n_masks} masks")
    assert n_images == n_masks, f"mismatch in {yolo_split}"

sample = sorted((OUTPUT_DIR / "images" / "train").glob("*.png"))[0]
mask_sample = OUTPUT_DIR / "masks" / "train" / sample.name
mask_arr = np.array(Image.open(mask_sample))
print(sample.name, "unique raw label ids:", sorted(np.unique(mask_arr).tolist()))

train: 1517 images, 1517 masks
val: 166 images, 166 masks
test: 392 images, 392 masks
site10_000001_012400.png unique raw label ids: [0, 9, 10, 12, 13, 14, 15, 17, 29, 30, 31, 36]


In [6]:
print("Train with:")
print(f'yolo semantic train data="{OUTPUT_DIR / "coralscapes.yaml"}" model=yolo26n-sem.pt epochs=100 imgsz=1024')

Train with:
yolo semantic train data="C:\Users\jordan\Documents\GitHub\coralscapes_yolo\coralscapes.yaml" model=yolo26n-sem.pt epochs=100 imgsz=1024
